# 03. PPO 학습 결과 분석

**목적**: PPO 1M 스텝 학습 과정과 결과를 분석한다.

**핵심 질문**:
- 학습이 수렴했는가?
- best_model과 final_model 중 어느 것이 낫는가?
- 베이스라인 대비 현재 성능은?

In [ ]:
import sys
sys.path.insert(0, '..')

import re
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

from src.utils.config import load_config
from src.agents.ppo_agent import PPOAgent
from src.evaluation.metrics import compute_all, print_metrics
from src.agents.baselines import run_all_baselines

import os
os.makedirs('../reports/semester1/figures', exist_ok=True)
print('라이브러리 로드 완료')

In [ ]:
config     = load_config('../config/experiment_config.yaml')
df_train   = pd.read_parquet('../data/processed/btc_train.parquet')
df_val     = pd.read_parquet('../data/processed/btc_val.parquet')
initial_cash = config['environment']['initial_cash']
print('데이터 로드 완료')

## 1. 학습 곡선 (Val Sharpe 추이)

In [ ]:
# 로그 파일 파싱
with open('../experiments/exp001_baseline/train_log.txt', encoding='utf-8', errors='replace') as f:
    log = f.read()

pattern = r'\[eval\] step=\s*([\d,]+) \| Sharpe=([+-]?\d+\.\d+) \| Return=([+-]?\d+\.\d+)% \| MDD=(\d+\.\d+)%'
matches = re.findall(pattern, log)

eval_df = pd.DataFrame(matches, columns=['step','sharpe','return_pct','mdd_pct'])
eval_df['step']       = eval_df['step'].str.replace(',','').astype(int)
eval_df['sharpe']     = eval_df['sharpe'].astype(float)
eval_df['return_pct'] = eval_df['return_pct'].astype(float)
eval_df['mdd_pct']    = eval_df['mdd_pct'].astype(float)

best_idx    = eval_df['sharpe'].idxmax()
best_step   = eval_df.loc[best_idx, 'step']
best_sharpe = eval_df.loc[best_idx, 'sharpe']

print(f'총 평가 포인트: {len(eval_df)}개')
print(f'최고 Sharpe: {best_sharpe:.3f} (step={best_step:,})')
print(f'최종 Sharpe: {eval_df["sharpe"].iloc[-1]:.3f} (step={eval_df["step"].iloc[-1]:,})')
eval_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('PPO 학습 곡선 (Val 셋 평가, exp001)', fontsize=13, fontweight='bold')

# 베이스라인 기준선
baseline_sharpe = 2.610  # Fixed Grid 1%

# Sharpe 추이
ax = axes[0]
ax.plot(eval_df['step'], eval_df['sharpe'], 'b-o', ms=4, lw=1.5, label='PPO Sharpe')
ax.axhline(baseline_sharpe, color='red', ls='--', lw=1.5, label=f'Fixed Grid 1% ({baseline_sharpe:.2f})')
ax.axhline(0, color='black', ls='-', lw=0.6, alpha=0.4)
ax.scatter([best_step], [best_sharpe], color='gold', s=100, zorder=5, label=f'Best ({best_sharpe:.3f})')
ax.set_title('Val Sharpe Ratio')
ax.set_xlabel('학습 스텝'); ax.set_ylabel('Sharpe')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

# 수익률 추이
ax = axes[1]
ax.plot(eval_df['step'], eval_df['return_pct'], 'g-o', ms=4, lw=1.5)
ax.axhline(0, color='black', ls='-', lw=0.6, alpha=0.4)
ax.set_title('Val 누적 수익률 (%)')
ax.set_xlabel('학습 스텝'); ax.set_ylabel('수익률 (%)')
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

# MDD 추이
ax = axes[2]
ax.plot(eval_df['step'], eval_df['mdd_pct'], 'r-o', ms=4, lw=1.5)
ax.set_title('Val MDD (%)')
ax.set_xlabel('학습 스텝'); ax.set_ylabel('MDD (%)')
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

plt.tight_layout()
plt.savefig('../reports/semester1/figures/03_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: 03_learning_curve.png')

## 2. best_model vs final_model 비교

In [ ]:
# best_model 평가
best_agent = PPOAgent.load('../experiments/exp001_baseline/best_model', config, df_train, df_val)
m_best = best_agent.evaluate(df_val)
print_metrics(m_best, label=f'PPO best_model (step={best_step:,})')

# final_model 평가
final_agent = PPOAgent.load('../experiments/exp001_baseline/final_model', config, df_train, df_val)
m_final = final_agent.evaluate(df_val)
print_metrics(m_final, label='PPO final_model (step=1,000,000)')

## 3. best_model 자본 곡선 vs 베이스라인

In [ ]:
bl_results = run_all_baselines(df_val, config)

fig, ax = plt.subplots(figsize=(14, 6))

# 베이스라인 (회색 계열)
gray_styles = {
    'buy_and_hold':    ('#2196F3', 1.5, '--', 'Buy-and-Hold'),
    'fixed_grid_1pct': ('#4CAF50', 1.5, '--', 'Fixed Grid 1%'),
    'atr_grid_k1.0':   ('#FF9800', 1.5, '--', 'ATR Grid k=1.0'),
}
for name, (c, lw, ls, lbl) in gray_styles.items():
    eq = bl_results[name]['equity_curve']
    ax.plot(eq.values, label=lbl, color=c, lw=lw, ls=ls, alpha=0.7)

# PPO best_model (굵게)
ax.plot(m_best['equity_curve'].values, label=f'PPO best (Sharpe={m_best["sharpe_ratio"]:.3f})',
        color='#9C27B0', lw=2.5, ls='-')

ax.axhline(initial_cash, color='black', lw=0.8, ls=':', alpha=0.5)
ax.set_title('PPO best_model vs 베이스라인 자본 곡선 (Val 셋, 2023)', fontsize=12, fontweight='bold')
ax.set_xlabel('스텝 (시간봉)'); ax.set_ylabel('포트폴리오 가치 ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/semester1/figures/03_ppo_vs_baselines.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: 03_ppo_vs_baselines.png')

## 4. 종합 비교표

In [ ]:
all_metrics = {}
for name, r in bl_results.items():
    all_metrics[name] = compute_all(r['equity_curve'], initial_cash, r['n_trades'], r['completed_cycles'])
all_metrics[f'PPO_best(step={best_step:,})'] = m_best
all_metrics['PPO_final(step=1M)']            = m_final

# equity_curve 키 제거 (출력용)
for k in all_metrics:
    all_metrics[k] = {kk: vv for kk, vv in all_metrics[k].items() if kk != 'equity_curve' and kk != 'completed_cycles'}

comp_df = pd.DataFrame(all_metrics).T
comp_df = comp_df[['total_return_pct','sharpe_ratio','max_drawdown_pct','n_trades','n_cycles']]
comp_df.columns = ['수익률(%)', 'Sharpe', 'MDD(%)', '거래수', '사이클수']
comp_df = comp_df.sort_values('Sharpe', ascending=False)
print(comp_df.round(3).to_string())

## 5. 결과 해석 및 향후 방향

### 관찰
- PPO best_model (100k 스텝): Sharpe +0.795 — 초기에 학습되다가
- 150k 스텝 이후 Sharpe 0.15~0.35 구간에서 진동, 수렴 실패
- final_model (1M 스텝): Sharpe -0.306 — 성능 하락

### 원인 분석 (가설)
1. **학습률 과다**: `lr=3e-4`가 후반부에 정책을 불안정하게 만들 수 있음 → lr scheduling 검토
2. **엔트로피 과다**: `ent_coef=0.01`이 수렴을 방해할 수 있음 → 감소 또는 annealing
3. **보상 스케일**: step_reward가 매우 작은 값 → 정규화 검토
4. **에피소드 길이**: Train 셋 1 에피소드 = 25,916스텝 → `n_steps=2048` 대비 매우 긴 horizon

### 2학기 튜닝 계획
- learning_rate scheduling (cosine decay)
- ent_coef annealing (0.01 → 0.001)
- reward clipping / normalization
- n_steps 증가 (2048 → 4096 또는 8192)
- 총 학습 스텝 증가 (1M → 5M)